In [4]:
import boto3
import json
import time
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

def invoke_model(bedrock, model_id, text, request_id):
    body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 100,
        "messages": [{"role": "user", "content": text}]
    }
    try:
        start = time.time()
        response = bedrock.invoke_model(modelId=model_id, body=json.dumps(body))
        data = json.loads(response['body'].read())
        latency = time.time() - start
        tokens = data['usage']['input_tokens'] + data['usage']['output_tokens']
        print(f"[{datetime.now().strftime('%H:%M:%S')}] {request_id}: SUCCESS ({tokens:,} tokens, {latency:.2f}s)")
        return (request_id, "SUCCESS", tokens)
    except Exception as e:
        print(f"[{datetime.now().strftime('%H:%M:%S')}] {request_id}: ERROR - {str(e)}")
        return (request_id, "ERROR", 0)

def test_tpm_quotas(throttle_model, throttle_name, other_model, other_name, region='us-east-1'):
    bedrock = boto3.client('bedrock-runtime', region_name=region)
    
    large_text = "This is a test document for Amazon Bedrock quota testing. " * 1000
    
    throttle_results = []
    other_results = []
    throttled = False
    
    print(f"\nSending large payloads to {throttle_name} and {other_name} over 120 seconds")
    print(f"Target: {throttle_name} exceeds 200K TPM, {other_name} stays under 180K TPM")
    print("="*80)
    
    start_time = time.time()
    end_time = start_time + 120
    
    with ThreadPoolExecutor(max_workers=50) as executor:
        futures = []
        throttle_count = 0
        other_count = 0
        
        while time.time() < end_time:
            for i in range(3):
                futures.append(executor.submit(
                    invoke_model, bedrock, throttle_model, large_text, f"{throttle_name}-{throttle_count}"
                ))
                throttle_count += 1
            
            if throttled:
                for i in range(3):
                    futures.append(executor.submit(
                        invoke_model, bedrock, other_model, large_text, f"{other_name}-{other_count}"
                    ))
                    other_count += 1
            
            time.sleep(5)
            
            completed = [f for f in futures if f.done()]
            for future in completed:
                req_id, status, tokens = future.result()
                if status == "ERROR" and throttle_name in req_id and not throttled:
                    throttled = True
                    print(f"\n*** {throttle_name} THROTTLED - Now sending to {other_name} ***\n")
        
        for future in as_completed(futures):
            request_id, status, tokens = future.result()
            if throttle_name in request_id:
                throttle_results.append((status, tokens))
            else:
                other_results.append((status, tokens))
    
    print("\n" + "="*80)
    print("RESULTS")
    print("="*80)
    
    throttle_success = sum(1 for s, t in throttle_results if s == "SUCCESS")
    throttle_error = sum(1 for s, t in throttle_results if s == "ERROR")
    throttle_total = len(throttle_results)
    throttle_tokens = sum(t for s, t in throttle_results if s == "SUCCESS")
    
    other_success = sum(1 for s, t in other_results if s == "SUCCESS")
    other_error = sum(1 for s, t in other_results if s == "ERROR")
    other_total = len(other_results)
    other_tokens = sum(t for s, t in other_results if s == "SUCCESS")
    
    print(f"\n{throttle_name}:")
    print(f"  Total requests: {throttle_total}")
    print(f"  Success: {throttle_success} ({throttle_success/throttle_total*100:.1f}%)")
    print(f"  Error: {throttle_error} ({throttle_error/throttle_total*100:.1f}%)")
    print(f"  Total tokens: {throttle_tokens:,}")
    
    print(f"\n{other_name}:")
    print(f"  Total requests: {other_total}")
    print(f"  Success: {other_success} ({other_success/other_total*100:.1f}%)")
    print(f"  Error: {other_error} ({other_error/other_total*100:.1f}%)")
    print(f"  Total tokens: {other_tokens:,}")

US_MODEL = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"
GLOBAL_MODEL = "global.anthropic.claude-sonnet-4-5-20250929-v1:0"

print("Which profile to throttle?")
print("1. US CRIS")
print("2. Global CRIS")
choice = input("Enter 1 or 2: ").strip()

if choice == "1":
    test_tpm_quotas(US_MODEL, "US-CRIS", GLOBAL_MODEL, "Global-CRIS")
else:
    test_tpm_quotas(GLOBAL_MODEL, "Global-CRIS", US_MODEL, "US-CRIS")

Which profile to throttle?
1. US CRIS
2. Global CRIS


Enter 1 or 2:  1



Sending large payloads to US-CRIS and Global-CRIS over 120 seconds
Target: US-CRIS exceeds 200K TPM, Global-CRIS stays under 180K TPM
[02:20:44] US-CRIS-0: SUCCESS (13,108 tokens, 3.34s)
[02:20:44] US-CRIS-1: SUCCESS (13,108 tokens, 3.63s)
[02:20:44] US-CRIS-2: SUCCESS (13,108 tokens, 3.73s)
[02:20:49] US-CRIS-4: SUCCESS (13,108 tokens, 3.16s)
[02:20:49] US-CRIS-3: SUCCESS (13,108 tokens, 3.52s)
[02:20:49] US-CRIS-5: SUCCESS (13,108 tokens, 3.56s)
[02:20:54] US-CRIS-7: SUCCESS (13,108 tokens, 3.57s)
[02:20:54] US-CRIS-6: SUCCESS (13,108 tokens, 3.70s)
[02:20:55] US-CRIS-8: SUCCESS (13,108 tokens, 3.97s)
[02:20:59] US-CRIS-10: SUCCESS (13,108 tokens, 3.43s)
[02:20:59] US-CRIS-11: SUCCESS (13,108 tokens, 3.53s)
[02:21:00] US-CRIS-9: SUCCESS (13,108 tokens, 3.76s)
[02:21:04] US-CRIS-13: SUCCESS (13,108 tokens, 3.31s)
[02:21:04] US-CRIS-12: SUCCESS (13,108 tokens, 3.34s)
[02:21:04] US-CRIS-14: SUCCESS (13,108 tokens, 3.57s)
[02:21:09] US-CRIS-15: SUCCESS (13,108 tokens, 3.51s)
[02:21:09] 